In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "adventure_works")
dbutils.widgets.text("schema_silver", "silver")
dbutils.widgets.text("schema_golden", "golden")

catalog = dbutils.widgets.get("catalog")
schema_silver = dbutils.widgets.get("schema_silver")
schema_golden = dbutils.widgets.get("schema_golden")


In [0]:
df_cust = spark.table(f"{catalog}.{schema_silver}.customer")
df_prod = spark.table(f"{catalog}.{schema_silver}.product")
df_hdr = spark.table(f"{catalog}.{schema_silver}.so_header")
df_det = spark.table(f"{catalog}.{schema_silver}.so_detail")
df_terr = spark.table(f"{catalog}.{schema_silver}.territory")


In [0]:
# Dimensión de producto
dim_product = df_prod.select(
    "product_id",
    "Name",
    "ProductNumber",
    "Color",
    "list_price",
    "standard_cost",
    "ProductLine",
    "Class",
    "Style"
)

# Dimensión de cliente
dim_customer = df_cust.select(
    "customer_id",
    "person_id",
    "store_id",
    "territory_id",
    "AccountNumber",
)


# Dimensión de territorio
dim_territory = df_terr.select(
    "territory_id",
    "Name",
    "CountryRegionCode",
    "territory_group"
)


# Consulta con join
fact_sales = (
    df_det.alias("d")
    .join(df_hdr.alias("h"), col("d.salesorder_id") == col("h.salesorder_id"))
    .join(df_prod.alias("p"), col("d.ProductID") == col("p.product_id"), "left")
    .select(
        col("d.salesorder_detail_id"),
        col("h.salesorder_id"),
        col("h.orderdate"),
        col("h.customerid").alias("customer_id"),
        col("d.productid").alias("product_id"),
        col("d.OrderQty").alias("quantity"),
        col("d.LineTotal").alias("line_total"),
        col("h.territoryid").alias("territory_id")
    )
)


In [0]:
dim_customer.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_golden}.dim_customer")
dim_product.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_golden}.dim_product")
dim_territory.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_golden}.dim_territory")
fact_sales.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema_golden}.fact_sales")

